# Preparação de Dados e Feature Engineering

**Atividade:** Divisão de dados · Data leakage · CRISP-DM · Encoding · Escalonamento · Transformações · Feature engineering · Seleção de variáveis  
**Dataset:** Painel de ações brasileiras (PETR4, VALE3, ITUB4, BBDC4, MGLU3, WEGE3) — 2015 a 2025  
**Fonte:** Yahoo Finance via `yfinance`

---

## Contexto

Em projetos de ciência de dados, a maior parte do trabalho — e dos erros — está **antes** do modelo. Um modelo simples sobre dados bem preparados quase sempre vence um modelo complexo sobre dados sujos. Este notebook percorre o pipeline de preparação na ordem em que ele aparece no dia a dia.

| Seção | Pergunta respondida |
|---|---|
| **CRISP-DM** | Qual é o roteiro padrão de um projeto de ciência de dados? |
| **Divisão de dados** | Como separar treino, validação e teste sem viciar o resultado? |
| **Data leakage** | O que pode contaminar meu pipeline e fazer o modelo parecer melhor do que é? |
| **Encoding** | Como transformo categóricas em números sem inventar ordem inexistente? |
| **Escalonamento** | Quando usar MinMax vs. StandardScaler? |
| **Transformações** | Como reduzir assimetria em variáveis enviesadas? |
| **Feature engineering** | Como criar variáveis que carregam o sinal que o modelo precisa? |
| **Seleção de variáveis** | Como cortar o que é ruído ou redundância? |

**Tarefa de exemplo:** prever se o retorno do dia seguinte de uma ação será positivo (classificação binária).

## 1. CRISP-DM — o pipeline conceitual

**CRISP-DM** (Cross-Industry Standard Process for Data Mining) é o roteiro mais usado em projetos de dados. Seis fases iterativas:

| Fase | O que se faz | Entregável típico |
|------|--------------|-------------------|
| **1. Business Understanding** | Traduzir o problema de negócio em problema de dados | Pergunta + métrica de sucesso |
| **2. Data Understanding** | Coletar e explorar (EDA) | Dicionário de dados + relatório de qualidade |
| **3. Data Preparation** | Limpar, transformar, criar features | Dataset modelável |
| **4. Modeling** | Escolher e treinar modelos | Modelos candidatos |
| **5. Evaluation** | Validar métrica e aderência ao negócio | Modelo aprovado |
| **6. Deployment** | Colocar em produção, monitorar | API/dashboard + plano de monitoria |

**Pontos importantes:**
- O fluxo **não é linear** — descobertas em *Modeling* fazem voltar para *Preparation* ou até para *Business Understanding*.
- A fase 3 costuma consumir **60–80%** do tempo de um projeto.
- Este notebook foca da fase 2 à fase 3.

## 2. Bibliotecas e dados

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import yfinance as yf

from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, RobustScaler,
    OneHotEncoder, LabelEncoder
)
from sklearn.feature_selection import (
    VarianceThreshold, mutual_info_classif, SelectKBest, f_classif
)
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import roc_auc_score, accuracy_score

import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
np.random.seed(42)

In [ ]:
# Painel de seis ações brasileiras com setor (categórica real para encoding)
ativos = {
    'PETR4.SA': 'Energia',
    'VALE3.SA': 'Materiais',
    'ITUB4.SA': 'Financeiro',
    'BBDC4.SA': 'Financeiro',
    'MGLU3.SA': 'Consumo',
    'WEGE3.SA': 'Industrial',
}

raw = yf.download(list(ativos.keys()), start='2015-01-01', end='2025-12-31',
                  auto_adjust=True, progress=False)

# Reorganiza para formato 'long' (uma linha por ticker × data)
df = (raw.stack(level=1, future_stack=True)
         .rename_axis(['Date', 'Ticker'])
         .reset_index())
df['Setor'] = df['Ticker'].map(ativos)
df = df.sort_values(['Ticker', 'Date']).reset_index(drop=True)
df.head()

## 3. Divisão de dados — treino, validação, teste e holdout

Dividir os dados é o ato fundador de qualquer projeto supervisionado. O objetivo é estimar a performance do modelo em dados que ele **não viu** — a única medida honesta de generalização.

### 3.1 Os três conjuntos

| Conjunto | Para que serve | Quando o modelo o vê |
|---|---|---|
| **Treino** | Estimar parâmetros do modelo | Durante o `fit()` |
| **Validação** | Ajustar hiperparâmetros, comparar candidatos | Várias vezes, no desenvolvimento |
| **Teste** | Estimar performance final | **Uma única vez**, no fim |

### 3.2 Holdout vs. validação cruzada

- **Holdout simples:** uma única divisão (ex.: 70/15/15). Rápido, mas a estimativa depende da divisão sorteada — alta variância em datasets pequenos.
- **K-Fold CV:** divide o treino em $k$ partes, treina $k$ vezes rodando a validação. Estimativa mais estável.
- **Stratified K-Fold:** preserva proporções de classes em cada fold (essencial para classes desbalanceadas).
- **TimeSeriesSplit:** para séries temporais — a validação é **sempre depois** do treino. Nunca prever passado a partir do futuro.

### 3.3 Proporções típicas

- Datasets pequenos (~1k linhas): 60/20/20 ou 5-fold CV + 20% holdout.
- Datasets médios (~100k): 70/15/15.
- Datasets grandes (>1M): 98/1/1 já é suficiente — 1% de 1M são 10k pontos de teste.

### 3.4 Erro clássico em séries temporais

Usar `train_test_split(shuffle=True)` em séries temporais **vaza informação do futuro para o treino** e infla artificialmente a métrica. Em séries, sempre dividir por **data de corte**.

In [ ]:
# Divisão temporal correta: corte por data
corte_treino = '2022-12-31'
corte_val    = '2024-06-30'

treino_idx = df['Date'] <= corte_treino
val_idx    = (df['Date'] > corte_treino) & (df['Date'] <= corte_val)
teste_idx  = df['Date'] > corte_val

print(f'Treino   : {treino_idx.sum():>7,}  ({df.loc[treino_idx, "Date"].min().date()} → {df.loc[treino_idx, "Date"].max().date()})')
print(f'Validação: {val_idx.sum():>7,}  ({df.loc[val_idx, "Date"].min().date()} → {df.loc[val_idx, "Date"].max().date()})')
print(f'Teste    : {teste_idx.sum():>7,}  ({df.loc[teste_idx, "Date"].min().date()} → {df.loc[teste_idx, "Date"].max().date()})')

In [ ]:
# TimeSeriesSplit ilustrativo — janelas crescentes
tscv = TimeSeriesSplit(n_splits=5)
exemplo = np.arange(20)
for i, (tr, te) in enumerate(tscv.split(exemplo)):
    print(f'Fold {i+1}: treino={list(tr)}  teste={list(te)}')

## 4. Data Leakage — como o futuro vaza para o passado

**Data leakage** é qualquer situação em que o modelo, durante o treino, recebe informação que **não estaria disponível no momento da predição real**. O sintoma é cruel: métricas excelentes em treino e validação, performance pífia em produção.

### 4.1 Tipos comuns

| Tipo | Exemplo | Como evitar |
|---|---|---|
| **Target leakage** | Variável que é uma função do alvo (ex.: usar `volume_do_dia_seguinte` para prever retorno) | Listar quando cada variável fica disponível |
| **Train/test contamination** | Aplicar `StandardScaler.fit()` no dataset todo antes de dividir | Sempre `fit` só no treino |
| **Look-ahead em séries** | Calcular média móvel usando dados futuros, ou shuffle em série temporal | Divisão temporal estrita |
| **Group leakage** | Mesmo cliente/ativo em treino e teste em problema de generalização para novos | `GroupKFold` |
| **Encoding leakage** | Target encoding ajustado no dataset todo | Ajustar dentro do CV |

### 4.2 Regra de ouro

> **Tudo que envolva *aprender algo dos dados* (estatísticas, encodings, imputação, escalonamento) deve ser ajustado SOMENTE no treino e aplicado nos demais.**

Em `sklearn`, isso é resolvido com `Pipeline` + `ColumnTransformer`: o pipeline ajusta no `fit()` (treino) e só transforma no `predict()` (teste).

In [ ]:
# Demonstração do risco: padronizar antes vs. depois de dividir
from sklearn.datasets import make_classification
X_demo, y_demo = make_classification(n_samples=500, n_features=10, random_state=0)

# ❌ ERRADO: padroniza com estatísticas de TODOS os dados
scaler_errado = StandardScaler().fit(X_demo)
X_errado = scaler_errado.transform(X_demo)
Xtr_e, Xte_e, ytr_e, yte_e = train_test_split(X_errado, y_demo, random_state=0)

# ✅ CERTO: divide primeiro, padroniza só com o treino
Xtr, Xte, ytr, yte = train_test_split(X_demo, y_demo, random_state=0)
scaler_certo = StandardScaler().fit(Xtr)
Xtr_c = scaler_certo.transform(Xtr)
Xte_c = scaler_certo.transform(Xte)

print('A diferença pode ser pequena em datasets simples — mas em produção,')
print('a versão errada usa estatísticas de TESTE que você não teria em tempo real.')

## 5. Construção do alvo e features básicas

**Alvo:** o retorno do dia seguinte é positivo? (1/0)

**Cuidado:** o `shift(-1)` puxa o futuro para a linha de hoje. Isso é correto na construção do alvo, mas seria leakage se aplicado a uma feature.

In [ ]:
df['Retorno']      = df.groupby('Ticker')['Close'].pct_change()
df['RetornoFuturo'] = df.groupby('Ticker')['Retorno'].shift(-1)
df['Alvo']          = (df['RetornoFuturo'] > 0).astype(int)

# Features iniciais — TUDO deve usar apenas dados até a linha atual
df['Volume_log'] = np.log1p(df['Volume'])
df['Amplitude']  = (df['High'] - df['Low']) / df['Close']
df.head()

## 6. Feature Engineering — criando variáveis derivadas

Feature engineering é onde o conhecimento de domínio entra no pipeline. Para cada feature criada, pergunte: **'isso usa só informação que eu teria no momento da predição?'**

### 6.1 Famílias de features em séries financeiras

| Família | Exemplos |
|---|---|
| **Lags** | retorno de ontem, anteontem, há 5 dias |
| **Janelas móveis** | média 5d, volatilidade 21d, máximo 60d |
| **Razões** | preço / média móvel, volume / média de volume |
| **Calendário** | dia da semana, mês, fim de mês |
| **Agregações por grupo** | retorno médio do setor no dia |

Toda janela móvel **DEVE** usar `rolling().shift(1)` ou equivalente para garantir que o valor da linha $t$ não inclui o próprio $t$ (quando é o caso) — ou pelo menos ter clareza do que está incluído.

In [ ]:
g = df.groupby('Ticker', group_keys=False)

# Lags do retorno (já estão no passado — seguros)
for lag in [1, 2, 5]:
    df[f'Ret_lag{lag}'] = g['Retorno'].shift(lag)

# Estatísticas em janela móvel — uso apenas o histórico, sem o dia atual
df['Ret_media_5']   = g['Retorno'].apply(lambda s: s.shift(1).rolling(5).mean())
df['Ret_media_21']  = g['Retorno'].apply(lambda s: s.shift(1).rolling(21).mean())
df['Vol_21d']       = g['Retorno'].apply(lambda s: s.shift(1).rolling(21).std())
df['Volume_med_21'] = g['Volume_log'].apply(lambda s: s.shift(1).rolling(21).mean())

# Razão preço / média móvel — proxy de momentum
df['MM_50']    = g['Close'].apply(lambda s: s.shift(1).rolling(50).mean())
df['Preco_MM'] = df['Close'] / df['MM_50']

# Features de calendário
df['DiaSemana'] = df['Date'].dt.day_name()
df['Mes']       = df['Date'].dt.month
df['FimMes']    = df['Date'].dt.is_month_end.astype(int)

# Agregação por grupo (setor, no dia) — característica do contexto, não do próprio ativo
df['RetSetor_dia'] = df.groupby(['Date', 'Setor'])['Retorno'].transform('mean')

df = df.dropna().reset_index(drop=True)
print(f'Linhas após dropna: {len(df):,}')
df.tail()

## 7. Encoding de variáveis categóricas

Modelos numéricos (regressão logística, redes neurais, gradient boosting) precisam de números. Mas a forma de transformar a categórica importa muito.

### 7.1 Label Encoding

Atribui um inteiro a cada categoria: `Energia=0, Materiais=1, ...`. **Cria uma ordem artificial** que o modelo pode tomar como real. Apropriado **apenas** quando a categoria é genuinamente ordinal (ex.: 'baixo' < 'médio' < 'alto') ou para árvores, que tratam splits sem ordem.

### 7.2 One-Hot Encoding

Cria uma coluna binária por categoria. Não inventa ordem. **Custo:** explode a dimensionalidade quando a categórica tem muitos níveis (alta cardinalidade — CEP, ID de cliente). Usar `drop='first'` em modelos lineares para evitar colinearidade perfeita.

### 7.3 Target Encoding

Substitui cada categoria pela **média do alvo** condicional àquela categoria. Mantém a dimensionalidade baixa e funciona bem com alta cardinalidade. **Risco:** é a forma de encoding mais propensa a leakage — a média DEVE ser calculada apenas no treino (idealmente com regularização ou dentro do CV).

### 7.4 Resumo de quando usar

| Encoder | Quando | Cuidado |
|---|---|---|
| Label | Ordinais; árvores | Não usar em lineares se não for ordinal |
| One-Hot | Baixa cardinalidade (<~20) | Explode em alta cardinalidade |
| Target | Alta cardinalidade | Risco de leakage; usar suavização |

In [ ]:
amostra = df[['Ticker', 'Setor', 'DiaSemana', 'Alvo']].head(10).copy()

# Label Encoding (atenção: cria ordem)
amostra['Setor_label'] = LabelEncoder().fit_transform(amostra['Setor'])

# One-Hot — pandas resolve em uma linha
one_hot = pd.get_dummies(amostra['Setor'], prefix='Setor').astype(int)
amostra = pd.concat([amostra, one_hot], axis=1)

amostra

In [ ]:
# Target encoding manual com SUAVIZAÇÃO — evita supervalorizar categorias raras
# Calculado APENAS no treino (uso o conjunto de treino temporal)
treino = df[df['Date'] <= corte_treino]

media_global = treino['Alvo'].mean()
alpha = 50  # força da suavização

stats_setor = treino.groupby('Setor')['Alvo'].agg(['mean', 'count'])
stats_setor['target_enc'] = (
    (stats_setor['mean'] * stats_setor['count'] + media_global * alpha)
    / (stats_setor['count'] + alpha)
)
stats_setor

## 8. Normalização e Padronização

Modelos sensíveis à escala (regressão linear/logística, KNN, SVM, redes neurais, PCA) exigem que as features estejam em escalas comparáveis. Modelos baseados em **árvores** (Random Forest, Gradient Boosting) **não** precisam — splits são invariantes a transformações monotônicas.

### 8.1 Os três escalonadores

| Método | Fórmula | Quando usar |
|---|---|---|
| **MinMaxScaler** (normalização) | $x' = \dfrac{x - x_{min}}{x_{max} - x_{min}}$ | Quando preciso intervalo fixo $[0,1]$; dados sem outliers extremos |
| **StandardScaler** (padronização) | $x' = \dfrac{x - \bar{x}}{s}$ | Default em quase tudo; assume distribuição aproximadamente simétrica |
| **RobustScaler** | $x' = \dfrac{x - \tilde{x}}{IQR}$ | Dados com outliers (séries financeiras!) |

**Regra de ouro novamente:** `fit()` só no treino.

In [ ]:
feat_continuas = ['Ret_media_5', 'Vol_21d', 'Volume_med_21', 'Preco_MM', 'Amplitude']
treino_X = df.loc[df['Date'] <= corte_treino, feat_continuas]

comparacao = pd.DataFrame({
    'Original'  : treino_X.iloc[:, 0],
    'MinMax'    : MinMaxScaler().fit_transform(treino_X)[:, 0],
    'Standard'  : StandardScaler().fit_transform(treino_X)[:, 0],
    'Robust'    : RobustScaler().fit_transform(treino_X)[:, 0],
})
comparacao.describe().round(3)

## 9. Transformações de variáveis

Variáveis muito **assimétricas** (skewed) violam pressupostos de modelos lineares e dão peso desproporcional aos valores grandes. Transformações monotônicas comprimem a cauda.

| Transformação | Aplicabilidade | Efeito |
|---|---|---|
| **Log** $\log(x)$ ou $\log(1+x)$ | $x > 0$ (use `log1p` se há zeros) | Forte compressão — recomendado para volume, riqueza, contagens |
| **Raiz quadrada** $\sqrt{x}$ | $x \geq 0$ | Compressão moderada — boa para contagens com poucos valores grandes |
| **Box-Cox** | $x > 0$ | Família parametrizada por $\lambda$; escolhe a 'melhor' transformação |
| **Yeo-Johnson** | qualquer $x$ | Generalização de Box-Cox que aceita negativos |

Como decidir? **Antes:** olhar histograma + skewness; **depois:** comparar skewness pós-transformação. Skewness próxima de 0 indica simetria.

In [ ]:
vol = df['Volume'].dropna()
transformacoes = {
    'Original' : vol,
    'Raiz'     : np.sqrt(vol),
    'Log'      : np.log1p(vol),
    'Box-Cox'  : pd.Series(stats.boxcox(vol[vol > 0])[0]),
}

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, (nome, serie) in zip(axes, transformacoes.items()):
    ax.hist(serie, bins=60, edgecolor='white')
    ax.set_title(f'{nome}\nskew={serie.skew():.2f}')
plt.tight_layout(); plt.show()

## 10. Seleção de variáveis

Mais features ≠ modelo melhor. Features irrelevantes ou redundantes:
- aumentam variância e risco de overfitting,
- diluem a importância das variáveis úteis,
- encarecem treino e inferência.

Três famílias de seleção:

### 10.1 Filtros (estatísticos)

Avaliam cada feature isoladamente, sem rodar o modelo. Rápidos, bons para um primeiro corte.

- **Variância baixa** (`VarianceThreshold`): variável quase constante carrega zero informação.
- **Correlação com o alvo** (Pearson, Spearman, mutual information).
- **Correlação entre features**: quando duas features são quase iguais ($|r| > 0{,}9$), mantém-se uma.

### 10.2 Wrappers

Treinam o modelo com diferentes subconjuntos. Custosos, mas levam em conta interações. Ex.: Recursive Feature Elimination (RFE).

### 10.3 Embedded (importância pelo próprio modelo)

- **Coeficientes em modelos lineares com regularização L1 (Lasso)** zeram features irrelevantes.
- **`feature_importances_`** em árvores e gradient boosting.
- **Permutation importance**: mede a queda de performance quando uma feature é embaralhada — interpretação mais robusta.

In [ ]:
feats = ['Ret_lag1', 'Ret_lag2', 'Ret_lag5', 'Ret_media_5', 'Ret_media_21',
         'Vol_21d', 'Volume_med_21', 'Preco_MM', 'Amplitude', 'RetSetor_dia',
         'FimMes', 'Mes']

X_tr = df.loc[df['Date'] <= corte_treino, feats]
y_tr = df.loc[df['Date'] <= corte_treino, 'Alvo']

# Filtro 1 — variância (após padronizar, para escalas comparáveis)
X_tr_std = StandardScaler().fit_transform(X_tr)
vt = VarianceThreshold(threshold=0.01).fit(X_tr_std)
print('Features que passam no filtro de variância:',
      [f for f, k in zip(feats, vt.get_support()) if k])

# Filtro 2 — correlação com o alvo (Spearman, robusta)
corr_alvo = X_tr.apply(lambda c: stats.spearmanr(c, y_tr).correlation)
corr_alvo.abs().sort_values(ascending=False).round(4)

In [ ]:
# Filtro 3 — redundância entre features
fig, ax = plt.subplots(figsize=(9, 6))
sns.heatmap(X_tr.corr(), annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax)
ax.set_title('Correlação entre features (treino)')
plt.show()

In [ ]:
# Mutual information — captura relações não-lineares com o alvo
mi = mutual_info_classif(X_tr, y_tr, random_state=0)
pd.Series(mi, index=feats).sort_values(ascending=False).round(4)

In [ ]:
# Embedded — importância de Random Forest
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=0, n_jobs=-1)
rf.fit(X_tr, y_tr)

imp = pd.Series(rf.feature_importances_, index=feats).sort_values()
imp.plot(kind='barh', figsize=(8, 5), color='steelblue')
plt.title('Importância de features — Random Forest')
plt.xlabel('importance'); plt.tight_layout(); plt.show()

## 11. Juntando tudo num Pipeline (sem leakage)

Pipeline e ColumnTransformer encadeiam pré-processamento e modelo num único objeto. Ao chamar `fit()`, **todas as estatísticas (médias, OHE, etc.) são aprendidas SOMENTE no treino**, e ao chamar `predict()` no teste, são apenas aplicadas. Esta é a defesa estrutural contra train/test contamination.

In [ ]:
feat_num = ['Ret_lag1', 'Ret_lag5', 'Ret_media_21', 'Vol_21d',
            'Volume_med_21', 'Preco_MM', 'Amplitude', 'RetSetor_dia']
feat_cat = ['Setor', 'DiaSemana']

X = df[feat_num + feat_cat]
y = df['Alvo']

X_tr, X_val, X_te = X[treino_idx], X[val_idx], X[teste_idx]
y_tr, y_val, y_te = y[treino_idx], y[val_idx], y[teste_idx]

preproc = ColumnTransformer([
    ('num', StandardScaler(), feat_num),
    ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), feat_cat),
])

pipe = Pipeline([
    ('preproc', preproc),
    ('modelo' , LogisticRegression(max_iter=1000)),
])

pipe.fit(X_tr, y_tr)
for nome, Xs, ys in [('Treino', X_tr, y_tr), ('Validação', X_val, y_val), ('Teste', X_te, y_te)]:
    proba = pipe.predict_proba(Xs)[:, 1]
    pred  = (proba >= 0.5).astype(int)
    print(f'{nome:>10s} → AUC={roc_auc_score(ys, proba):.4f}  ACC={accuracy_score(ys, pred):.4f}')

## 12. Síntese

| Etapa | Ferramentas-chave | Armadilhas |
|---|---|---|
| **CRISP-DM** | 6 fases iterativas | Pular *Business Understanding* |
| **Divisão** | holdout, K-Fold, TimeSeriesSplit | shuffle em série temporal |
| **Leakage** | `Pipeline`, `ColumnTransformer` | `fit` antes do split |
| **Encoding** | LabelEncoder, OneHot, Target | Target encoding fora do CV |
| **Escalonamento** | StandardScaler, MinMax, Robust | Aplicar em árvores (desnecessário) |
| **Transformações** | log, sqrt, Box-Cox, Yeo-Johnson | log em variável com zeros (use `log1p`) |
| **Feature engineering** | lags, rolling, agregações | Janelas que incluem o presente |
| **Seleção** | variância, correlação, MI, importância | Selecionar usando o conjunto de teste |

> **Mensagem final:** o pipeline de dados é uma máquina de blindagem. Cada `Pipeline` bem montado, cada `fit` feito apenas no treino, cada janela móvel que respeita a flecha do tempo é uma camada de honestidade que separa um modelo que funciona em produção de um que só performa no slide.